<a href="https://colab.research.google.com/github/VakitiSriVarsha/Gen-AI/blob/main/RAG_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers sentence-transformers faiss-cpu

In [ ]:
documents = [
    "Retrieval Augmented Generation combines information retrieval with large language models.",
    "FAISS is used for efficient similarity search on vector embeddings.",
    "Sentence Transformers convert text into numerical embeddings.",
    "RAG improves factual accuracy by grounding LLMs with external documents.",
    "Google Colab is a cloud-based platform for running Python notebooks."
]

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

embedder = SentenceTransformer("all-MiniLM-L6-v2")
doc_embeddings = embedder.encode(documents)
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(doc_embeddings))

In [ ]:
def retrieve_context(query, k=2):
    query_embedding = embedder.encode([query])
    distances, indices = index.search(np.array(query_embedding), k)

    retrieved_docs = [documents[i] for i in indices[0]]
    return retrieved_docs, distances[0]

In [ ]:
from transformers import pipeline

llm = pipeline(
    "text-generation",
    model="gpt2",
    pad_token_id=50256
)

In [ ]:
def rag_answer(query, threshold=1.0):
    context, distances = retrieve_context(query)

    if min(distances) < threshold:
        prompt = f"""
Use the context below to answer the question.

Context:
{context}

Question:
{query}

Answer:
"""
    else:
        prompt = f"""
Answer the question using your general knowledge.

Question:
{query}

Answer:
"""

    response = llm(
        prompt,
        max_new_tokens=120,
        do_sample=True,
        temperature=0.7
    )[0]["generated_text"]

    return response

In [ ]:
questions = [
    "What is Retrieval Augmented Generation?",
    "Why is FAISS used?",
    "What is Sentence Transformer?",
    "What is Google Colab?",
    "Who is the Prime Minister of India?"
]

for q in questions:
    print("Q:", q)
    print("A:", rag_answer(q))
    print("-" * 50)